# Per-model PCA + clustering -- each backend gets its own embedding, one cell per model

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

`model_capability_exploration.ipynb` fits **one shared** 2-D PCA per protein
on all 6 ABCfold backends pooled together, then just filters/colours a given
backend's points within that shared space -- good for "does backend X land
somewhere the others don't", not for "what would backend X find on its own".

This notebook asks the second question: for each backend, a **fresh PCA is
fit using only that backend's own frames** for a protein (not the pooled
ensemble), then clusters that backend-only embedding -- GMM-auto (BIC-knee,
same default as `tm_conformation_clustering.ipynb`) unless `cluster_method`
is set to `'hdbscan'` (DBCV/Optuna-tuned, also mirroring that notebook).
Two backends can end up with completely different numbers of clusters and
different axes -- that's the point, each model's conformational landscape is
assessed entirely on its own terms.

A single backend can genuinely defeat GMM here: a model that returns a lot of
near-duplicate frames (e.g. RosettaFold3 writing both a `_model` and a
`_model_fixed` CIF per sample) can make a GMM component's covariance
ill-conditioned once the BIC-knee sweep tries enough components -- HDBSCAN
doesn't require well-defined per-cluster covariance, so `cluster_method='hdbscan'`
is the fallback for a backend where GMM keeps failing.

One markdown + code cell pair per backend (AlphaFold3, Boltz-2, Chai-1,
OpenFold3, Protenix, RosettaFold3); each code cell loops over every protein
discovered under `results/tm_alignment/` at run time (small-multiples grid,
one panel per protein with enough frames from that backend), so **no new
cells are needed as more proteins land** -- just re-run top to bottom.

Every cluster's CIFs are symlinked into
`results/tm_reannotated/<protein>/<backend>_pca_<method_tag>/cluster_<i>/`
(same mechanism as `tm_conformation_clustering.ipynb`'s `_reannotate`), so a
given backend's clusters can be loaded straight into ChimeraX.


In [1]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import optuna
from hdbscan.validity import validity_index
from kneed import KneeLocator
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one INFO line per trial is too noisy at hdbscan_n_trials=40/call

ROOT             = Path("..")
ABCFOLD_OUT_ROOT = ROOT / "results" / "abcfold"
ALIGN_ROOT       = ROOT / "results" / "tm_alignment"
REANN_ROOT       = ROOT / "results" / "tm_reannotated"

GMM_PALETTE = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#a65628", "#f781bf", "#999999",
]

# Default HDBSCAN search space for cluster_method='hdbscan' -- same
# candidate values as tm_conformation_clustering.ipynb, applied to a
# single backend's own 2-D embedding here instead of the pooled one.
HDBSCAN_MIN_SAMPLES_CANDIDATES = [3, 5, 10, 15, 20, 25, 30]
HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
HDBSCAN_CLUSTER_SELECTION_METHODS = ["eom", "leaf"]
HDBSCAN_METRICS = ["euclidean", "cityblock"]

# Mirrors scripts/tm_helix_alignment.py's BACKEND_PATTERNS -- duplicated
# here (not imported) so this notebook stays self-contained, same
# convention tm_conformation_clustering.ipynb uses. Keep in sync if the
# script's version changes.
BACKEND_PATTERNS = {
    "alphafold3":   "alphafold3",
    "boltz":        "boltz",
    "chai1":        "chai",
    "openfold3":    "openfold",
    "protenix":     "protenix",
    "rosettafold3": "rosettafold",
}


def _load_run(run_name):
    """Load one apo/holo run's aligned TM-Ca ensemble + per-frame metadata
    (model/backend, seed, sample index, pTM, RMSD-to-mean), written by
    scripts/tm_helix_alignment.py -- already pooled across every one of
    ABCfold's 6 backends x seed x diffusion/sample for this run."""
    npy = ALIGN_ROOT / run_name / "aligned_ca_tm.npy"
    csv = ALIGN_ROOT / run_name / "meta.csv"
    if not npy.exists():
        raise FileNotFoundError(
            f"{npy} not found -- run worflows/postprocessing/Snakefile "
            f"(scripts/tm_helix_alignment.py) for {run_name} first")
    coords = np.load(npy)                          # (n_frames, n_ca_tm, 3)
    meta   = pd.read_csv(csv)
    X      = coords.reshape(coords.shape[0], -1)   # flatten to (n_frames, n_ca_tm*3)
    return X, meta


def _kabsch_fit(P, Q):
    """Rotation R (3,3) and translation t (3,) such that (R @ P.T).T + t ~= Q.
    Same as kabsch() in scripts/tm_helix_alignment.py."""
    p_mean, q_mean = P.mean(axis=0), Q.mean(axis=0)
    Pc, Qc = P - p_mean, Q - q_mean
    U, _, Vt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    t = q_mean - R @ p_mean
    return R, t


def _load_protein(protein):
    """Load the aligned, multi-backend TM-Ca ensemble for one BASE protein
    (e.g. 'NPF2.12_Q9LFX9'), merging its apoform and holoform ABCfold runs
    into a single pooled ensemble -- same logic as
    tm_conformation_clustering.ipynb's _load_protein (holo's mean TM
    structure Kabsch-refit onto apo's before pooling, since apo and holo are
    independent ABCfold jobs with no shared global orientation). Backend-only
    subsets are sliced out of this pooled ensemble per (protein, backend)
    below -- see _backend_pca_cluster -- rather than loaded separately per
    backend, so each protein's npy/csv are only read once.

    Adds 'status' ('apo'/'holo'), 'source_run' and 'unique_frame_id' columns
    to meta, needed by _reannotate to find the right CIFs."""
    X_parts, meta_parts = [], []
    apo_mean = None
    for status in ("apo", "holo"):
        run_name = f"{protein}__{status}"
        if not (ALIGN_ROOT / run_name).exists():
            if status == "apo":
                raise FileNotFoundError(
                    f"{ALIGN_ROOT / run_name} not found -- apoform is expected "
                    f"for every protein; run worflows/postprocessing/Snakefile first")
            continue
        X, meta = _load_run(run_name)
        coords = X.reshape(X.shape[0], -1, 3)  # (n_frames, n_ca_tm, 3)

        if status == "apo":
            apo_mean = coords.mean(axis=0)
        else:
            R, t = _kabsch_fit(coords.mean(axis=0), apo_mean)
            flat = coords.reshape(-1, 3)
            coords = ((R @ flat.T).T + t).reshape(coords.shape)
            X = coords.reshape(coords.shape[0], -1)

        meta = meta.copy()
        meta["status"] = status
        meta["source_run"] = run_name
        meta["unique_frame_id"] = status + "_" + meta["frame_id"].astype(str)
        X_parts.append(X)
        meta_parts.append(meta)

    X    = np.concatenate(X_parts, axis=0)
    meta = pd.concat(meta_parts, ignore_index=True)
    return X, meta


def _backend_of(path, predictions_dir):
    try:
        top = path.relative_to(predictions_dir).parts[0].lower()
    except (ValueError, IndexError):
        return "unknown"
    for backend, pattern in BACKEND_PATTERNS.items():
        if pattern in top:
            return backend
    return "unknown"


def _discover_abcfold_cifs(run_name):
    """Every model CIF ABCfold produced for one apo/holo run. Mirrors
    scripts/tm_helix_alignment.py's discover_predictions()."""
    predictions_dir = ABCFOLD_OUT_ROOT / run_name
    return sorted(c for c in predictions_dir.rglob("*.cif") if "templates" not in c.parts)


def _frame_id_for_cif(cif_path, predictions_dir):
    """Same frame_id derivation as scripts/tm_helix_alignment.py's
    parse_frame_id(), so a rediscovered CIF can be matched back to its
    meta.csv row by (source_run, frame_id)."""
    rel   = cif_path.relative_to(predictions_dir)
    model = _backend_of(cif_path, predictions_dir)
    m = re.search(r"seed-?(\d+)_sample-?(\d+)", str(rel), re.IGNORECASE)
    if m:
        return f"{model}_seed{m.group(1)}_sample{m.group(2)}"
    return f"{model}_{rel.with_suffix('')}".replace("/", "_")


def _ellipse_trace(mean, cov, color, n_std=1.5, n_pts=80):
    vals, vecs = np.linalg.eigh(cov)
    idx        = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    t          = np.linspace(0, 2 * np.pi, n_pts)
    pts        = n_std * (vecs * np.sqrt(np.maximum(vals, 0))) @ np.vstack(
                     [np.cos(t), np.sin(t)])
    x, y = mean[0] + pts[0], mean[1] + pts[1]
    return go.Scatter(
        x=np.append(x, x[0]), y=np.append(y, y[0]),
        mode="lines",
        line=dict(color=color, width=1.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    )


def _reannotate(protein, meta, labels, x_col, y_col, method_tag,
                max_per_cluster=20, sample_seed=42):
    """Symlink each structure's CIF into
    results/tm_reannotated/<protein>/<method_tag>/cluster_<k>/. Same logic as
    tm_conformation_clustering.ipynb's _reannotate -- CIFs are looked up by
    (source_run, frame_id) since apo and holo runs independently repeat the
    same model/seed/sample numbering."""
    cif_by_key = {}
    for source_run in sorted(meta["source_run"].unique()):
        predictions_dir = ABCFOLD_OUT_ROOT / source_run
        for c in _discover_abcfold_cifs(source_run):
            cif_by_key[(source_run, _frame_id_for_cif(c, predictions_dir))] = c

    meta = meta.copy()
    meta["gmm_cluster"] = labels
    out_dir     = REANN_ROOT / protein / method_tag
    cluster_ids = sorted(set(int(l) for l in labels))

    assign_rows = []
    n_symlinked = 0
    for cid in cluster_ids:
        dir_name    = "cluster_noise" if cid == -1 else f"cluster_{cid}"
        cluster_dir = out_dir / dir_name
        cluster_dir.mkdir(parents=True, exist_ok=True)
        for stale in cluster_dir.iterdir():
            if stale.is_symlink():
                stale.unlink()

        cluster_rows = meta[meta["gmm_cluster"] == cid]
        sampled_idx = set(cluster_rows.sample(
            n=min(len(cluster_rows), max_per_cluster), random_state=sample_seed,
        ).index)

        for idx, row in cluster_rows.iterrows():
            cif = cif_by_key.get((row["source_run"], row["frame_id"]))
            symlinked = cif is not None and idx in sampled_idx
            if symlinked:
                dest = cluster_dir / f"{row['unique_frame_id']}.cif"
                if not dest.exists():
                    dest.symlink_to(cif.resolve())
                    n_symlinked += 1
            assign_rows.append({
                "protein":      protein,
                "status":       row["status"],
                "model":        row["model"],
                "seed":         row["seed"],
                "sample_index": row["sample_index"],
                "frame_id":     row["unique_frame_id"],
                "ptm":          row["ptm"],
                "cluster":      cid,
                x_col:          round(float(row[x_col]), 4),
                y_col:          round(float(row[y_col]), 4),
                "symlinked":    symlinked,
            })
    pd.DataFrame(assign_rows).to_csv(out_dir / "assignments.csv", index=False)
    print(f"  [reannotate] {protein}/{method_tag}: {n_symlinked} symlinks "
          f"(max {max_per_cluster}/cluster) of {len(assign_rows)} assignments -> {out_dir}")


def _fit_gmm_bic_sweep(xy, k_min=1, k_max=20, n_init=20, random_state=42, reg_covar=1e-5):
    """Fit a GaussianMixture for every k in [k_min, k_max] and return the
    model sitting at the knee of the BIC-vs-k curve. Same approach as
    tm_conformation_clustering.ipynb's _fit_gmm_bic_sweep: a polynomial fit
    through the BIC curve (kneed, degree capped relative to the number of
    k's swept) smooths out single noisy restarts before finding the elbow,
    falling back to the raw BIC minimum if KneeLocator finds no knee.

    Fitting an all-backend-pooled embedding (as tm_conformation_clustering.ipynb
    does) rarely produces exact/near-duplicate points; a single backend's own
    frames can (e.g. RosettaFold3 writes both a '_model' and a '_model_fixed'
    CIF per sample -- near-identical coordinates), which can make a component's
    empirical covariance singular once the sweep tries enough components. Each
    k is fit in its own try/except and simply dropped from the sweep (not
    fatal to the whole cell) if that happens; reg_covar is bumped above
    sklearn's 1e-6 default as a first line of defense. Raises if every k in
    the sweep fails (try cluster_method='hdbscan' instead in that case)."""
    xy = np.asarray(xy, dtype=np.float64)
    k_max = min(k_max, xy.shape[0] - 1)
    ks    = list(range(max(1, k_min), k_max + 1))

    gmms, bic_by_k = {}, {}
    for k in ks:
        try:
            gmm = GaussianMixture(n_components=k, covariance_type="full",
                                   n_init=n_init, random_state=random_state,
                                   reg_covar=reg_covar)
            gmm.fit(xy)
        except ValueError as e:
            print(f"[gmm-auto] k={k} failed (ill-conditioned covariance), skipping: {e}")
            continue
        gmms[k]     = gmm
        bic_by_k[k] = float(gmm.bic(xy))

    if not gmms:
        raise RuntimeError(
            f"GMM failed to fit for every k in {ks} -- this backend's frames are "
            f"likely too degenerate (many exact/near-duplicate structures) for a "
            f"covariance-based fit. Try cluster_method='hdbscan' instead.")

    best_k = min(bic_by_k, key=bic_by_k.get)
    ks_ok  = sorted(bic_by_k)
    if len(ks_ok) >= 3:
        degree = min(7, max(1, len(ks_ok) - 3))
        try:
            kl = KneeLocator(ks_ok, [bic_by_k[k] for k in ks_ok],
                              curve="convex", direction="decreasing",
                              interp_method="polynomial", polynomial_degree=degree)
            if kl.knee is not None:
                best_k = int(kl.knee)
        except Exception as e:
            print(f"[gmm-auto] WARNING: KneeLocator failed ({e}), falling back to BIC minimum")

    return gmms[best_k], best_k, bic_by_k


def _fit_hdbscan_dbcv_search(xy, min_samples_candidates=HDBSCAN_MIN_SAMPLES_CANDIDATES,
                              min_cluster_size_candidates=HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES,
                              cluster_selection_methods=HDBSCAN_CLUSTER_SELECTION_METHODS,
                              metrics=HDBSCAN_METRICS, n_trials=40, random_state=42):
    """Optuna/TPE search over (min_samples, min_cluster_size,
    cluster_selection_method, metric), scored by DBCV (Moulavi et al. 2014)
    via hdbscan.validity.validity_index -- same approach as
    tm_conformation_clustering.ipynb's _fit_hdbscan_dbcv_search, applied to a
    single backend's own 2-D embedding here. Combos yielding fewer than 2
    clusters, or that error inside DBCV, score -1.0 so they're never
    selected."""
    xy = np.asarray(xy, dtype=np.float64)
    n = xy.shape[0]
    max_min_cluster_size = max(2, n // 5)
    candidate_min_cluster_sizes = [m for m in min_cluster_size_candidates if 2 <= m <= max_min_cluster_size]
    if not candidate_min_cluster_sizes:
        candidate_min_cluster_sizes = [max_min_cluster_size]

    grid_size = (len(min_samples_candidates) * len(candidate_min_cluster_sizes)
                 * len(cluster_selection_methods) * len(metrics))
    n_trials = min(n_trials, grid_size)

    def objective(trial):
        min_samples = trial.suggest_categorical("min_samples", list(min_samples_candidates))
        min_cluster_size = trial.suggest_categorical("min_cluster_size", candidate_min_cluster_sizes)
        cluster_selection_method = trial.suggest_categorical("cluster_selection_method", list(cluster_selection_methods))
        metric = trial.suggest_categorical("metric", list(metrics))
        try:
            labels = HDBSCAN(min_samples=min_samples, min_cluster_size=min_cluster_size,
                              cluster_selection_method=cluster_selection_method,
                              metric=metric, copy=False).fit(xy).labels_
            n_clust = len(set(c for c in labels if c >= 0))
            dbcv = float(validity_index(xy, labels, metric=metric)) if n_clust >= 2 else -1.0
        except Exception as e:
            print(f"[hdbscan-auto] combo ms={min_samples} mcs={min_cluster_size} "
                  f"{cluster_selection_method}/{metric} failed: {e}")
            labels, dbcv = None, -1.0
        trial.set_user_attr("labels", None if labels is None else labels.tolist())
        return dbcv

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_trial  = study.best_trial
    best_labels = best_trial.user_attrs["labels"]
    best = {
        "min_samples": best_trial.params["min_samples"],
        "min_cluster_size": best_trial.params["min_cluster_size"],
        "cluster_selection_method": best_trial.params["cluster_selection_method"],
        "metric": best_trial.params["metric"],
        "dbcv": best_trial.value,
    }
    labels = np.array(best_labels) if best_labels is not None else np.full(n, -1)
    return labels, best


print("Setup done.")


Setup done.


## Protein discovery

Every protein under `results/tm_alignment/` at run time, apo/holo suffix
stripped back to the BASE name `_load_protein` expects.

In [2]:
def _base_protein_name(dirname):
    if dirname.endswith("__apo") or dirname.endswith("__holo"):
        return dirname.rsplit("__", 1)[0]
    return dirname


PROTEINS = sorted({
    _base_protein_name(p.name) for p in ALIGN_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith(".")
})

print(f"{len(PROTEINS)} protein(s) under results/tm_alignment/: {PROTEINS}")


1 protein(s) under results/tm_alignment/: ['NPF2.12_Q9LFX9']


## Precompute raw ensembles

Load each protein's pooled (all 6 backends, apo+holo Kabsch-aligned) raw
TM-Ca coordinates + metadata **once** here, cached in `PROTEIN_RAW`. Every
per-backend cell below slices its own backend's rows out of this cache and
fits a fresh PCA on just that slice (see `_backend_pca_cluster`) -- so a
protein's npy/csv are only read from disk once no matter how many backend
cells use it, even though every backend still gets its own independent
embedding.

In [3]:
PROTEIN_RAW = {}  # protein -> (X, meta) -- full pooled ensemble, all backends

for protein in PROTEINS:
    X, meta = _load_protein(protein)
    PROTEIN_RAW[protein] = (X, meta)
    per_backend = ", ".join(f"{m}({n})" for m, n in meta["model"].value_counts().items())
    print(f"  {protein:20s} {len(meta):4d} frames total  [{per_backend}]")


  NPF2.12_Q9LFX9       4621 frames total  [openfold3(4000), rosettafold3(220), alphafold3(101), boltz(100), chai1(100), protenix(100)]


## Per-backend PCA + clustering

`_backend_pca_cluster(protein, backend, cluster_method="gmm", ...)` -- slices
`backend`'s own rows out of `PROTEIN_RAW[protein]`, fits a **fresh 2-D PCA
using only those rows** (not the pooled, all-backend embedding), then
clusters that embedding:

- `cluster_method="gmm"` (default) -- GMM-auto, BIC-knee (`_fit_gmm_bic_sweep`).
  Raises if every k in the sweep fails (a backend with enough
  exact/near-duplicate frames can make every component's covariance
  singular) -- retry that call with `cluster_method="hdbscan"`.
- `cluster_method="hdbscan"` -- DBCV/Optuna-tuned (`_fit_hdbscan_dbcv_search`),
  doesn't need well-defined per-cluster covariance so it tolerates duplicate
  frames; points HDBSCAN calls noise (`-1`) are kept unclustered rather than
  dropped.

Returns `None` if this backend has fewer than `min_frames` frames for this
protein (too few to fit a meaningful embedding). Symlinks every cluster's
CIFs into `results/tm_reannotated/<protein>/<backend>_pca_<method_tag>/cluster_<i>/`
(see `_reannotate`).

`plot_backend_clustering(backend, cluster_method="gmm", ...)` -- the cell
every per-backend section below calls: runs `_backend_pca_cluster` for every
protein, then renders a small-multiples grid (one panel per protein with
enough frames), each panel coloured by that protein's own local cluster
assignment (GMM: covariance ellipses; HDBSCAN: noise drawn as grey `x`
markers) -- cluster colours/ids are **not** comparable across panels (each
protein's clustering for this backend is entirely independent).

In [4]:
def _backend_pca_cluster(protein, backend, cluster_method="gmm", pc_x=1, pc_y=2,
                          auto_k_min=1, auto_k_max=20, hdbscan_n_trials=40,
                          max_per_cluster=20, min_frames=10):
    X, meta = PROTEIN_RAW[protein]
    mask = (meta["model"] == backend).to_numpy()
    n = int(mask.sum())
    if n < min_frames:
        return None

    X_sub = X[mask].astype(np.float64)  # GMM covariance is numerically fragile in float32
    meta_sub = meta[mask].reset_index(drop=True)
    n_pc = min(max(pc_x, pc_y, 2), X_sub.shape[1])
    pca = PCA(n_components=n_pc)
    coords = pca.fit_transform(X_sub)
    evr = pca.explained_variance_ratio_
    xy = coords[:, [pc_x - 1, pc_y - 1]]

    if cluster_method == "gmm":
        gmm, k_used, _ = _fit_gmm_bic_sweep(xy, k_min=auto_k_min, k_max=auto_k_max)
        labels = gmm.predict(xy)
        cluster_ids = list(range(k_used))
        means, covariances = gmm.means_, gmm.covariances_
        method_tag = f"gmm_k{k_used}"
    elif cluster_method == "hdbscan":
        labels, best = _fit_hdbscan_dbcv_search(xy, n_trials=hdbscan_n_trials)
        cluster_ids = sorted(c for c in set(labels) if c >= 0)
        k_used = len(cluster_ids)
        means, covariances = None, None
        method_tag = f"hdbscan_mcs{best['min_cluster_size']}"
    else:
        raise ValueError(f"cluster_method must be 'gmm' or 'hdbscan', got {cluster_method!r}")

    meta_sub = meta_sub.copy()
    meta_sub["pc_x"], meta_sub["pc_y"] = xy[:, 0], xy[:, 1]
    _reannotate(protein, meta_sub, labels, x_col="pc_x", y_col="pc_y",
                method_tag=f"{backend}_pca_{method_tag}", max_per_cluster=max_per_cluster)

    return {"meta": meta_sub, "labels": labels, "cluster_ids": cluster_ids,
            "means": means, "covariances": covariances, "k_used": k_used,
            "evr": evr, "n_frames": n}


def plot_backend_clustering(backend, cluster_method="gmm", pc_x=1, pc_y=2,
                             auto_k_min=1, auto_k_max=20, hdbscan_n_trials=40,
                             max_per_cluster=20, min_frames=10,
                             marker_size=6, opacity=0.7):
    """Independent per-protein PCA + clustering fit on ONLY `backend`'s own
    frames, one small-multiples panel per protein.

    backend  one of BACKEND_PATTERNS ('alphafold3', 'boltz', 'chai1',
             'openfold3', 'protenix', 'rosettafold3').
    cluster_method  'gmm' (default, BIC-knee auto) or 'hdbscan'
             (DBCV/Optuna-tuned auto) -- see _backend_pca_cluster. Switch to
             'hdbscan' for a backend where GMM raises (ill-conditioned
             covariance from duplicate/near-duplicate frames).
    min_frames  proteins with fewer than this many frames from `backend`
             are skipped (too few to fit a meaningful embedding/clustering).
    """
    results = {}
    for protein in PROTEINS:
        r = _backend_pca_cluster(protein, backend, cluster_method=cluster_method,
                                  pc_x=pc_x, pc_y=pc_y, auto_k_min=auto_k_min, auto_k_max=auto_k_max,
                                  hdbscan_n_trials=hdbscan_n_trials,
                                  max_per_cluster=max_per_cluster, min_frames=min_frames)
        if r is None:
            print(f"[{backend}] {protein}: fewer than {min_frames} frames, skipped")
            continue
        results[protein] = r

    if not results:
        print(f"No protein has >= {min_frames} {backend} frames yet.")
        return

    proteins_with_data = list(results)
    ncols = min(3, len(proteins_with_data))
    nrows = math.ceil(len(proteins_with_data) / ncols)
    subplot_titles = [f"{p}  (k={results[p]['k_used']}, n={results[p]['n_frames']})"
                      for p in proteins_with_data]
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=subplot_titles,
                        horizontal_spacing=0.08, vertical_spacing=0.12)

    legend_seen = set()
    for i, protein in enumerate(proteins_with_data):
        row, col = i // ncols + 1, i % ncols + 1
        r = results[protein]
        meta_sub, labels = r["meta"], r["labels"]
        for j, k in enumerate(r["cluster_ids"]):
            sub = meta_sub[labels == k]
            color = GMM_PALETTE[j % len(GMM_PALETTE)]
            show_legend = k not in legend_seen
            legend_seen.add(k)
            fig.add_trace(go.Scatter(
                x=sub["pc_x"], y=sub["pc_y"], mode="markers",
                marker=dict(size=marker_size, color=color, opacity=opacity),
                name=f"cluster {k}", legendgroup=f"cluster{k}", showlegend=show_legend,
                text=[f"seed {s} sample {si}" for s, si in zip(sub["seed"], sub["sample_index"])],
                hovertemplate="%{text}<extra></extra>",
            ), row=row, col=col)
            if r["means"] is not None:
                fig.add_trace(_ellipse_trace(r["means"][k], r["covariances"][k], color),
                              row=row, col=col)

        noise = meta_sub[labels == -1]
        if not noise.empty:
            fig.add_trace(go.Scatter(
                x=noise["pc_x"], y=noise["pc_y"], mode="markers",
                marker=dict(size=marker_size - 1, color="#aaa", opacity=0.4, symbol="x"),
                name="noise (HDBSCAN)", legendgroup="noise", showlegend=("noise" not in legend_seen),
            ), row=row, col=col)
            legend_seen.add("noise")

    fig.update_layout(
        title=f"{backend}<br>independent per-protein PCA + {cluster_method} clustering",
        template="plotly_white", height=650 * nrows, width=700 * ncols,
        legend_title="cluster (not comparable across panels)",
    )
    fig.show()


## Per-backend cells

One markdown + code cell pair per ABCfold backend. Each code cell is a
single `plot_backend_clustering(...)` call -- edit its arguments directly
(`cluster_method='hdbscan'` if GMM raises for that backend, `min_frames`,
`auto_k_min`/`auto_k_max`, `max_per_cluster`, `pc_x`/`pc_y`) for one-off
exploration on that backend without touching any other cell.

### `alphafold3`

Google DeepMind's AlphaFold3 -- the diffusion-based architecture the other 5 backends here are either reimplementations of or successors to; treated as the baseline.

In [5]:
plot_backend_clustering("alphafold3")

  [reannotate] NPF2.12_Q9LFX9/alphafold3_pca_gmm_k3: 26 symlinks (max 20/cluster) of 101 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/alphafold3_pca_gmm_k3


### `boltz`

Boltz-2 (MIT) -- open, AF3-style diffusion model with an additional binding-affinity head.

In [6]:
plot_backend_clustering("boltz")

  [reannotate] NPF2.12_Q9LFX9/boltz_pca_gmm_k2: 40 symlinks (max 20/cluster) of 100 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/boltz_pca_gmm_k2


### `chai1`

Chai-1 (Chai Discovery) -- open, AF3-style diffusion model trained independently of AlphaFold3.

In [7]:
plot_backend_clustering("chai1")

  [reannotate] NPF2.12_Q9LFX9/chai1_pca_gmm_k1: 20 symlinks (max 20/cluster) of 100 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/chai1_pca_gmm_k1


### `openfold3`

OpenFold3 -- fully open reimplementation and retraining of the AF3 architecture.

In [8]:
plot_backend_clustering("openfold3")

  [reannotate] NPF2.12_Q9LFX9/openfold3_pca_gmm_k3: 60 symlinks (max 20/cluster) of 4000 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/openfold3_pca_gmm_k3


### `protenix`

Protenix (ByteDance) -- open reimplementation of the AF3 architecture.

In [9]:
plot_backend_clustering("protenix")

  [reannotate] NPF2.12_Q9LFX9/protenix_pca_gmm_k3: 60 symlinks (max 20/cluster) of 100 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/protenix_pca_gmm_k3


### `rosettafold3`

RoseTTAFold3 (Baker lab) -- independent architecture lineage (RoseTTAFold), not an AF3 reimplementation.

In [10]:
plot_backend_clustering("rosettafold3")

  [reannotate] NPF2.12_Q9LFX9/rosettafold3_pca_gmm_k1: 20 symlinks (max 20/cluster) of 220 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/rosettafold3_pca_gmm_k1
